In [ ]:
import numpy as np
from src.LQG_controller import compute_lqr_gain, compute_kalman_gain
from src.simulate_lqg import simulate_lqg, simulate_robust_lqg
from src.sdp_solver import solve_sdp

# --- Parameters ---
T = 30.0
dt = 1.0
N = int(T / dt)

# In case we take a multidimensional system these constants have to be transformed in matrices and vectors
n = 1 # Number of states
m = 1 # Number of inputs
theta = [0.15] # has to be transformed into a diagonal matrix of shape n*n
theta = np.array(theta).reshape((n, 1))
mu = [130.0] * np.ones((n, 1)) # Mean of the state
x0 = [120.0] * np.ones((n, 1)) # Initial state

sigma_x0 = [2.0] # Initial state covariance
sigma_x0 = np.array(sigma_x0).reshape((n, 1))
sigma_w = [2.0] # Process noise
sigma_w = np.array(sigma_w).reshape((n, 1))
sigma_v = [2.0] # Measurement noise
sigma_v = np.array(sigma_v).reshape((m, 1))
# --- Cost Matrices ---
Q = 10 * np.eye(n) # State cost
R = 15 * np.eye(m) # Control cost

a = 1 - theta * dt
b = -dt
c = 1.0

A = np.array([[a]])
B = np.array([[b]])
C = np.array([[c]])

# --- LQG Gain ---
K, _ = compute_lqr_gain(A, B, Q, R)
Qw = sigma_w**2 * dt # Process noise covariance
Rv = sigma_v**2 # Measurement noise covariance
L, _ = compute_kalman_gain(A, C, Qw, Rv, sigma_x0)

# --- Robust Gains from SDP ---
X0_opt, W0_opt, V0_opt = solve_sdp(A, B, C, sigma_x0, sigma_w, sigma_v, dt, int(T/dt))

L_rob, _ = compute_kalman_gain(A, C, W0_opt[0][0], V0_opt[0][0], X0_opt[0][0])



In [ ]:
# --- Simulate ---
x, *_ = simulate_lqg(N, a, b, c, mu, K, L, sigma_x0, sigma_w, sigma_v, dt, x0)
x_rob, *_ = simulate_robust_lqg(N, a, b, c, mu, K, L_rob, W0_opt[0][0], V0_opt[0][0], dt, x0)



In [ ]:
# --- Plot Results (optional) ---
import matplotlib.pyplot as plt
plt.plot(x, label="LQG")
plt.plot(x_rob, label="Robust LQG")
plt.legend()
plt.title("LQG vs Robust LQG")
plt.show()
